# §28 — Cubic: KESİN TEST (n=16, eşleşmiş, ön-kayıtlı)

§27b'de cubic 6 seed'in 5'inde exp'i geçti (+0.411 nat) ama p≈0.10 — güç sorunu.
Gözlenen etki büyüklüğüyle (d≈0.82) **n=16 → t≈3.3 (p≈0.005)** olur *eğer etki
gerçekse*; değilse ayrışmaz. Yani soru ucuza (CPU) cevaplanabilir.

**Birincil endpoint (önceden sabit):** eğitim-sonu cross-chunk doğrulama kaybı.
**Eşleştirme:** aynı seed = aynı veri sırası → eşleşmiş t-testi geçerli.
**NaN kuralı:** ıraksayan koşu o kolun BAŞARISIZLIĞI sayılır, atılmaz (§27b dersi).

Kriterler RESULTS §28'de. Tamamen CPU. Seed 0-2 zaten var (cache'ten atlanır).


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, math
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
ROOT = os.path.join(BASE,'eta_sweep')          # §27 ile AYNI dizin -> seed 0-2 cache'i kullanilir
os.makedirs(ROOT, exist_ok=True)
N_SEEDS = 16
ARMS = [('cubic_flux_chunked','cubic_eta_default'), ('exp','exp_reference')]
print('repo:', REPO, '| cikti:', ROOT, '| n =', N_SEEDS)

In [ ]:
# --- 2. KOSU (seed 0..15, iki kol; tamamlananlar atlanir) ---
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO,
            'CC_CARRY_MAX':'16','CC_STEPS':'1200','CC_CTX':'256','CC_P':'6',
            'CC_DIST_EVERY':'64','CC_BS':'8','CC_GAPS':'256','CC_TRIALS':'60'}
RE_VER = re.compile(r'cross-chunk dogrulama loss:\s*([0-9.]+)')
RE_ACC = re.compile(r"FINAL acc%/logprob.*?\{256:\s*\(([0-9.]+)")
data={}
for mode, tag in ARMS:
    ck=os.path.join(ROOT,tag); os.makedirs(ck,exist_ok=True)
    env={**BASE_ENV,'HFP_CKPT_DIR':ck}
    env.pop('HFP_ETA_LOG_MIN',None); env.pop('HFP_ETA_LOG_MAX',None)   # varsayilan eta
    data[tag]={}
    for s in range(N_SEEDS):
        cache=os.path.join(ck,f'result_s{s}.json')
        if os.path.exists(cache):
            d=json.load(open(cache)); data[tag][s]=d; continue
        print(f'[{tag} s{s}] ...', end=' ', flush=True)
        r=subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',mode,str(s),'6000'],
                         cwd=REPO, env=env, capture_output=True, text=True)
        out=r.stdout+r.stderr
        mv=RE_VER.search(out); ma=RE_ACC.search(out)
        if mv:
            d={'loss':float(mv.group(1)), 'acc':float(ma.group(1)) if ma else None, 'nan':False}
            print(f'loss {d["loss"]:.3f}', flush=True)
        else:
            d={'loss':None,'acc':None,'nan':True}      # IRAKSAMA = basarisizlik, atilmaz
            print('IRAKSADI (NaN)', flush=True)
        json.dump(d, open(cache,'w')); data[tag][s]=d
json.dump(data, open(os.path.join(ROOT,'decisive_n16.json'),'w'), indent=2)
print('\nKOSU TAMAM')

In [ ]:
# --- 3. ON-KAYITLI HUKUM (§28): eslesmis t-testi ---
import statistics as st, math
C=data['cubic_eta_default']; E=data['exp_reference']
nan_c=[s for s in C if C[s]['nan']]; nan_e=[s for s in E if E[s]['nan']]
pairs=[(s, E[s]['loss'], C[s]['loss']) for s in range(N_SEEDS)
       if not C[s]['nan'] and not E[s]['nan']]
print(f'Iraksama: cubic {len(nan_c)} seed {nan_c} | exp {len(nan_e)} seed {nan_e}')
print(f'Gecerli eslesmis cift: {len(pairs)}/{N_SEEDS}\n')
d=[e-c for _,e,c in pairs]                       # pozitif = cubic daha iyi
cub=[c for _,_,c in pairs]; exp=[e for _,e,_ in pairs]
md_=st.mean(d); sd=st.stdev(d); n=len(d)
t=md_/(sd/math.sqrt(n)); wins=sum(1 for x in d if x>0)
# iki yonlu p (t-dagilimi, normal yaklasimi degil)
try:
    from statistics import NormalDist
    from math import lgamma
    def tcdf(t,df):   # regularized incomplete beta ile
        x=df/(df+t*t)
        def betacf(a,b,x):
            MAXIT,EPS,FPMIN=200,3e-12,1e-300
            qab,qap,qam=a+b,a+1,a-1
            c=1.0; dd=1-qab*x/qap
            if abs(dd)<FPMIN: dd=FPMIN
            dd=1/dd; h=dd
            for m in range(1,MAXIT+1):
                m2=2*m
                aa=m*(b-m)*x/((qam+m2)*(a+m2))
                dd=1+aa*dd; c=1+aa/c
                if abs(dd)<FPMIN: dd=FPMIN
                if abs(c)<FPMIN: c=FPMIN
                dd=1/dd; h*=dd*c
                aa=-(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
                dd=1+aa*dd; c=1+aa/c
                if abs(dd)<FPMIN: dd=FPMIN
                if abs(c)<FPMIN: c=FPMIN
                dd=1/dd; de=dd*c; h*=de
                if abs(de-1)<EPS: break
            return h
        a,b=df/2,0.5
        bt=math.exp(lgamma(a+b)-lgamma(a)-lgamma(b)+a*math.log(x)+b*math.log(1-x))
        ib=bt*betacf(a,b,x)/a if x<(a+1)/(a+b+2) else 1-math.exp(lgamma(a+b)-lgamma(a)-lgamma(b)+b*math.log(1-x)+a*math.log(x))*betacf(b,a,1-x)/b
        return ib          # = P(|T|>t)
    p=tcdf(abs(t),n-1)
except Exception as ex:
    p=float('nan'); print('p hesabi basarisiz:',ex)
print(f'cubic {st.mean(cub):.3f} | exp {st.mean(exp):.3f} | fark {md_:+.3f} nat')
print(f'cubic {wins}/{n} seed\'de daha iyi | eslesmis t={t:.2f} df={n-1} | iki yonlu p={p:.4f}\n')
print('=== ON-KAYITLI HUKUM (§28) ===')
if p<0.05 and wins>=0.7*n and md_>=0.20:
    print(f'CUBIC AVANTAJI DOGRULANDI: p={p:.4f}<0.05, {wins}/{n} seed (>=%70), fark {md_:.3f}>=0.20 nat.')
    print('  -> Seyrek-yazim rejiminde cubic ustunlugu BU OLCEKTE KANITLANDI. Rejime-ozgu mimari')
    print('     bulgu olarak belgelenir ve makale katki listesine girer.')
elif p>=0.05 and md_<0.15:
    print(f'REDDEDILDI: p={p:.4f}, fark {md_:.3f}<0.15 nat -> §26b/§27 egilimi gurultuymus.')
    print('  -> Cubic kendi ev rejiminde bile guvenilir avantaj vermiyor; bayrak olarak emekli edilir.')
else:
    print(f'HALA GUCSUZ: p={p:.4f}, fark {md_:.3f}, {wins}/{n} seed.')
    print('  -> Etki muhtemelen gercek ama tahminden kucuk. Boyle raporlanir; ek seed YOK (azalan getiri).')